# 毕昇工具链实验

## 1.1 实验概览

按顺序运行下面的代码，需要修改的参数已在代码中标出。


### 先打开实验文件

在 CANNLab 的终端下载完整仓库：

```bash
git clone https://gitcode.com/cann/cann-learning-hub.git
```

在文件树中打开 `contrib/tutorials/parallel_programming_principles_and_practice`，按各章 README 的顺序打开 Notebook。本章先打开 `01_bisheng_toolchain/01_lab.ipynb`，选择 `Python 3.11.4` 内核。

点击代码格左侧的运行按钮，从上到下执行。`%%bash` 表示整格运行 Bash，`!命令` 表示执行一条终端命令，`%env` 设置当前 Notebook 的环境变量。它们直接保留在 Notebook 中；在终端手动操作时不输入这些前缀。

文件路径以 Notebook 所在文件夹为起点。Python 变量和 `%env` 设置会延续到后续单元格；`%%bash` 中的 `cd` 只影响当前格。第 06、09 章的子实验从各自的 Notebook 运行，章节练习已写好子目录路径。


<a id="s-01-02_cann_development_environment"></a>


## 1.2 CANN 开发环境

### 本节学习目标

- 确认 CANN 版本和设备可见性
- 理解环境变量而非固定路径的定位方式

### 开始前

在 CANNLab 中选择 `Python 3.11.4` 内核。


自行搭建 Jupyter 时，先在终端加载 CANN 安装目录中的 `set_env.sh`，再从同一终端启动 `jupyter lab`。


### 主机与设备怎样配合

CPU 运行主机程序，准备数据并调用 ACL 运行库；NPU 执行设备算子。CANN 提供编译器、运行库和算子工具，Ascend C 用于编写设备计算代码。

本课程平台是 Atlas A3，实测 SoC 名称为 `Ascend910_9362`。源码中还保留 910B3 示例和 RTC 编译目标 `dav-2201` 的名称；当前课程已按本章所列环境运行。程序使用可见逻辑设备 `0`，不要直接用 `npu-smi` 表中的物理卡号或 Phy-ID 替换它。


### 查看 CANN 和设备信息

运行下面的单元格，查看 CANN、编译工具和 NPU。`npu-smi` 中的设备健康状态应为 OK。


In [ ]:
%%bash
set -e
echo "CANN 目录：$ASCEND_HOME_PATH"
cat "$ASCEND_HOME_PATH/compiler/version.info"
bisheng --version
cmake --version
npu-smi info


### 预期现象与结果分析

在正确的 CANN 环境中，应能定位 Toolkit 变量和相关工具；具体路径随镜像或安装方式变化。`npu-smi info` 应显示设备。

### 课后实践

选择一个工具，说明缺少它会影响后面的哪个实验。

参考答案见 `answer/01.02_answer.md`。


<a id="s-01-03_bisheng_compiler_basics"></a>


## 1.3 毕昇编译器基础

### 本节学习目标

- 区分 clang/clang++ 与 bisheng 的使用场景
- 识别主机入口和设备函数的声明

### 两种编译器

鲲鹏毕昇提供 `clang`/`clang++`，用于主机程序；CANN 的 `bisheng` 用于 Ascend C 源码。Ascend C 的目标架构参数按实际设备选择。


### 安装鲲鹏毕昇 Host 编译器

本课程使用毕昇 Host 3.2.0（ARM64）。下面从[华为云镜像](https://mirrors.huaweicloud.com/kunpeng/archive/compiler/bisheng_compiler/)下载并解压，约需 1 GiB。

已安装时跳过此单元格。离线机房由教师提前解压到 `~/opt/BiShengCompiler-3.2.0-aarch64-linux`；安装在其他位置时，修改下一步的 `compiler`。


In [ ]:
%%bash
set -e
mkdir -p ~/opt/BiShengCompiler-3.2.0-aarch64-linux
cd ~/opt/BiShengCompiler-3.2.0-aarch64-linux
URL=https://mirrors.huaweicloud.com/kunpeng/archive/compiler/bisheng_compiler
curl -fLO --connect-timeout 30 --retry 3 "$URL/BiShengCompiler-3.2.0-aarch64-linux.tar.gz"
tar -xzf BiShengCompiler-3.2.0-aarch64-linux.tar.gz --strip-components=1


### 设置 Host 编译器路径

`PATH` 指定命令的查找位置，`LD_LIBRARY_PATH` 指定动态库的查找位置。运行下面的单元格，让本 Notebook 后续命令使用刚安装的编译器。


In [ ]:
import os
from pathlib import Path

# 若安装在其他位置，只修改这一行。
compiler = Path.home() / "opt/BiShengCompiler-3.2.0-aarch64-linux"
os.environ["PATH"] = f"{compiler}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{compiler}/lib:{compiler}/lib/aarch64-unknown-linux-gnu:" + os.environ.get("LD_LIBRARY_PATH", "")


### 查看编译器版本

检查 `clang`、`clang++` 的版本输出是否包含 BiSheng 标识，再查看 CANN 的 `bisheng` 版本。


In [ ]:
%%bash
set -e
clang --version
clang++ --version
bisheng --version


### 阅读 Ascend C 示例

下面显示 hello_world.asc。分别找到设备函数和 main，留意设备函数的声明方式。

In [ ]:
!cat src/hello_world.asc


### 检查结果

`clang` 和 `clang++` 的版本输出应包含 BiSheng 标识。


### 课后实践

阅读 `src/hello_world.asc`，找到设备函数和主机 `main()`，指出它们的声明有什么不同。本节先识别两类代码，设备 Kernel 的编译与启动在第 06 章完成。

参考答案见 `answer/01.03_answer.md`。


<a id="s-01-04_host_cpp_build_workflow"></a>


## 1.4 Host C++ 构建流程

### 本节学习目标

- 使用鲲鹏毕昇 clang++ 构建 C++17 程序
- 区分 Host 编译与 AI Core 编译

### 不同程序使用哪些编译器

`clang++` 生成 Host C++ 可执行文件，`mpicxx` 负责 MPI Host 工程；`bisheng` 用于 Ascend C/AI Core 源。ACL 应用由 Host 编译器编译并链接 CANN 库。

In [ ]:
%%bash
set -e
mkdir -p build
clang++ -std=c++17 -O2 src/hello_host.cpp -o build/hello_host
./build/hello_host


### 预期现象与结果分析

程序应输出 `host toolchain ready`。这个任务不需要 NPU，可以用于区分主机编译问题和 CANN 运行问题。

### 课后实践

分别写出普通 C++、OpenMP、MPI wrapper、ACL 应用和 Ascend C Kernel 的编译方式。

参考答案见 `answer/01.04_answer.md`。

<a id="s-01-05_hpc_toolchain_dependencies"></a>


## 1.5 HPC 工具链依赖

### 本节学习目标

- 列出后续实验需要的编译器、库和设备
- 查看这些工具是否已经安装

### 依赖关系

OpenMP 依赖 Host 编译器及运行库；第 02 章 的构建脚本明确使用鲲鹏毕昇 `clang++`。MPI 依赖 `mpicxx/mpirun`；ACL 依赖 Toolkit 头文件和运行库；Ascend C 依赖 CANN `bisheng` 与目标架构；HCCL 还依赖设备、通信库和 rank/device 配置。

### 遇到问题时

找不到命令时检查安装和路径；编译报错时检查头文件与库；运行报错时检查设备和通信配置。


### 查看后续实验需要的工具

`which` 显示每个命令的位置。缺少的命令不会显示路径，需要在对应章节开始前安装。


In [ ]:
!which clang++ cmake mpicxx mpirun bisheng npu-smi


### 检查结果

缺少的工具在对应章节开始前安装，库和设备配置在各章编译运行时检查。


### 课后实践

任选一个后续实验，说明需要哪些工具、库和设备。

参考答案见 `answer/01.05_answer.md`。
